In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [8]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [9]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('selic/selic_diario.csv').set_index('date')

In [27]:
## SCORE para score
score_todos_anos = pd.read_csv('score_mf/magic_formula.csv').set_index('Unnamed: 0').reset_index()

In [28]:
score_todos_anos.rename(columns={'Unnamed: 0': 'date'}, inplace=True)

In [29]:
score_todos_anos.set_index('date', inplace=True)

In [30]:
lista_ativos_finais = score_todos_anos.columns.tolist()
score_todos_anos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015,0.884615,0.307692,0.641026,0.064103,0.692308,0.897436,0.307692,0.307692,0.307692,0.012821,...,0.948718,0.307692,0.769231,0.730769,0.051282,0.076923,0.307692,0.666667,0.564103,0.307692
2016,0.897436,0.243590,0.538462,0.833333,0.666667,0.602564,0.243590,0.243590,0.243590,1.000000,...,0.923077,0.243590,0.782051,0.679487,0.089744,0.551282,0.243590,0.730769,0.576923,0.243590
2017,0.910256,0.185897,0.551282,0.461538,0.628205,0.858974,0.185897,0.185897,0.185897,0.961538,...,0.820513,0.602564,0.576923,0.564103,0.371795,0.589744,0.185897,0.666667,0.474359,0.185897
2018,0.871795,0.173077,0.320513,0.756410,0.576923,0.884615,0.173077,0.173077,0.173077,0.961538,...,0.897436,0.538462,0.551282,0.346154,0.333333,0.602564,0.173077,0.730769,0.512821,0.173077
2019,0.833333,0.198718,0.448718,0.551282,0.679487,0.923077,0.198718,0.198718,0.198718,1.000000,...,0.807692,0.525641,0.512821,0.320513,0.333333,0.294872,0.198718,0.666667,0.602564,0.730769
2020,0.782051,0.179487,0.269231,0.525641,0.730769,0.794872,0.179487,0.179487,0.179487,0.961538,...,0.935897,0.500000,0.641026,0.358974,0.589744,0.602564,0.179487,0.564103,0.538462,0.307692
2021,0.717949,0.102564,0.269231,0.538462,0.794872,0.756410,0.102564,0.102564,0.102564,0.974359,...,0.987179,0.025641,0.384615,0.243590,0.807692,0.782051,0.410256,0.448718,0.487179,0.320513
2022,0.756410,0.115385,0.371795,0.307692,0.666667,0.743590,0.115385,0.115385,0.115385,0.987179,...,0.833333,0.025641,0.384615,0.615385,0.538462,0.717949,0.551282,0.448718,0.564103,0.410256
2023,0.794872,0.935897,0.538462,0.448718,0.628205,0.769231,0.096154,0.096154,0.096154,0.987179,...,0.833333,0.166667,0.384615,0.589744,0.179487,0.564103,0.730769,0.423077,0.576923,0.461538


## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [10]:
dict_sigma = {}
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]

    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['valor_diario']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))

        print(f"============\n SIGMA {ano}\n===========")            
        dict_sigma[ano] = dict_excesso[ano].cov()
        print('Tamanho final do SIGMA: ',len(dict_sigma[ano]))

    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2025
Tamanho final do SIGMA:  78
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  122
tamanho final do Excesso:  122
 SIGMA 2024
Tamanho final do SIGMA:  78
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  121
tamanho final do Excesso:  121
 SIGMA 2023
Tamanho final do SIGMA:  78
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  124
tamanho final do Excesso:  124
 SIGMA 2022
Tamanho final do SIGMA:  77
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2021
Tamanho final do SIGMA:  77
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  120
tamanho final do Excesso:  120
 SIGMA 2020
Tamanho final do SIGMA:  73
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic

## Hiperparâmetros

In [30]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [33]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [34]:
score_usado = score_todos_anos[score_todos_anos.index.isin([2015])]
score_usado

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015,0.884615,0.307692,0.641026,0.064103,0.692308,0.897436,0.307692,0.307692,0.307692,0.012821,...,0.948718,0.307692,0.769231,0.730769,0.051282,0.076923,0.307692,0.666667,0.564103,0.307692


In [35]:
dfs[str(2015)]

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015-10-01,0.014422,0.0,-0.044388,-0.009515,-0.012563,0.027080,-0.019078,0.016511,0.013532,0.010072,...,-0.016253,0.0,-0.021235,0.009731,-0.068671,0.038606,0.0,0.010755,0.028462,0.0
2015-10-02,0.018269,0.0,0.049697,0.076929,0.056493,0.035154,0.067740,0.042402,0.037756,0.064242,...,0.034113,0.0,0.040008,0.026389,0.012834,0.034842,0.0,-0.004170,0.028330,0.0
2015-10-05,0.014459,0.0,0.048363,0.033920,-0.019753,-0.019531,0.015084,0.023189,0.029319,0.010875,...,0.023205,0.0,0.023792,-0.003320,0.044310,0.017948,0.0,0.031597,0.024474,0.0
2015-10-06,-0.022605,0.0,0.044153,-0.036258,0.029484,-0.000893,0.022890,0.012116,0.009495,0.000750,...,0.015108,0.0,-0.016559,-0.003624,0.006028,0.009377,0.0,-0.006250,-0.002379,0.0
2015-10-07,0.005535,0.0,0.173864,0.012532,0.002387,0.002604,0.084700,0.034743,0.035919,0.048920,...,0.005947,0.0,0.035936,0.001453,0.021088,0.100494,0.0,0.021375,0.008367,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-03-24,0.000000,0.0,0.004588,0.000000,0.000000,0.002633,-0.033112,-0.023490,-0.018317,-0.011093,...,-0.011778,0.0,-0.005948,0.000720,0.039780,0.065857,0.0,0.002879,-0.009273,0.0
2016-03-28,0.015784,0.0,-0.050002,0.023053,0.036363,0.027387,0.058457,0.032869,0.048744,0.052561,...,0.016070,0.0,0.018628,0.000861,0.016446,0.007975,0.0,0.023233,0.023747,0.0
2016-03-29,0.002077,0.0,-0.026808,-0.021126,0.006578,-0.012064,0.029846,0.022305,0.017434,0.023637,...,0.022950,0.0,-0.018287,0.016047,-0.021527,-0.005265,0.0,0.001273,0.026002,0.0


In [27]:
dict_sigma[2015]

,ABEV3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,BEEF3,...,SLCE3,SMTO3,SUZB3,TAEE11,TOTS3,UGPA3,USIM5,VALE3,VIVT3,WEGE3
ABEV3,0.000339,0.000090,0.000346,0.000241,0.000246,0.000269,0.000257,0.000312,0.000331,0.000143,...,0.000016,0.000057,0.000112,0.000203,0.000204,0.000256,0.000100,0.000481,0.000209,0.000243
ANIM3,0.000090,0.002270,0.000487,0.000350,0.000336,0.000874,0.000606,0.000698,0.000370,0.000077,...,0.000188,0.000100,0.000169,0.000393,0.000206,0.000160,0.001193,0.001335,0.000573,0.000132
AXIA3,0.000346,0.000487,0.001414,0.000472,0.000711,0.001278,0.000783,0.000844,0.000883,0.000326,...,-0.000173,0.000122,-0.000285,0.000305,0.000211,0.000380,0.001505,0.000740,0.000422,0.000518
AZZA3,0.000241,0.000350,0.000472,0.000909,0.000370,0.000636,0.000407,0.000546,0.000401,0.000069,...,-0.000130,-0.000002,-0.000082,0.000116,0.000219,0.000229,0.000999,0.000765,0.000291,0.000246
B3SA3,0.000246,0.000336,0.000711,0.000370,0.000871,0.001038,0.000676,0.000722,0.000725,0.000183,...,-0.000040,0.000010,-0.000252,0.000212,0.000126,0.000288,0.001309,0.000739,0.000274,0.000195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UGPA3,0.000256,0.000160,0.000380,0.000229,0.000288,0.000364,0.000322,0.000344,0.000401,0.000168,...,0.000117,0.000020,0.000149,0.000179,0.000133,0.000353,0.000194,0.000492,0.000274,0.000282
USIM5,0.000100,0.001193,0.001505,0.000999,0.001309,0.002550,0.001151,0.001241,0.001406,0.000590,...,-0.000338,-0.000117,-0.001161,0.000325,-0.000127,0.000194,0.006672,0.001682,0.000693,0.000057
VALE3,0.000481,0.001335,0.000740,0.000765,0.000739,0.000987,0.000889,0.001034,0.000647,0.000492,...,0.000143,-0.000139,0.000355,0.000247,0.000414,0.000492,0.001682,0.003306,0.000926,0.000548
VIVT3,0.000209,0.000573,0.000422,0.000291,0.000274,0.000512,0.000314,0.000352,0.000345,0.000321,...,0.000152,0.000019,0.000249,0.000193,0.000214,0.000274,0.000693,0.000926,0.000820,0.000320


In [53]:
score_usado.dropna(axis=1)

,ABEV3,ANIM3,AXIA3,AZZA3,B3SA3,BBSE3,BEEF3,BRAP4,BRKM5,CMIG4,...,SBSP3,SLCE3,SUZB3,TAEE11,TOTS3,UGPA3,USIM5,VALE3,VIVT3,WEGE3
ano,,,,,,,,,,,,,,,,,,,,,
2015,0.594862,0.179014,-0.280358,0.188981,0.657607,-9.966455,0.180942,-4.348976,0.170975,8.511375,...,43.455029,0.133358,0.122797,0.778584,0.244284,0.20629,-0.293338,-0.134539,0.182352,0.125224


In [63]:
anos

['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31']

In [59]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:

            #SCORE ------------

        score_todos_anos = pd.read_csv(f'score_mf/{ano}/mf_{ano}.csv').set_index('ano')
        score_usado = score_todos_anos.dropna(axis=1).copy()
        # print(score_usado)
        print("Score Atualizado")
        ano_um = int(ano)

            # RETORNO ---------------

        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)]
        retorno_usado = df_usado.copy()
        retorno_usado = retorno_usado[score_usado.columns]

            # Lista ativos por ano ---------
        lista_ativos_finais[ano_um] = score_usado.columns.tolist()
        print("Retornos atualizados")

            # EXCESSO retorno - rf ------------

        excesso_usado = dict_excesso[ano_um][score_usado.columns]
        print("Excessos Atualizados")

            # SIGMA ---------------

        sigma_usado = excesso_usado.cov()
        print("Sigmas Atualizados")
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)
    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)
    print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)
    print(len(retorno_usado.columns))
    print(len(score_usado.columns))
    print(len(sigma_usado.columns))
    print(len(excesso_usado.columns))

    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    model.s = pyo.Param(initialize = 2, mutable=True)
    model.r = pyo.Var(within=pyo.NonNegativeReals,bounds=(0,10))
    #-------------------------------------- FUNÇÕES
    #=============================
    # Função Objetivo
    #=============================
    def func_objetivo_1(model):
        retorno_esperado = model.theta * sum(
            model.retornos_ativos[dia, a] * model.x[a] for a in model.ativos for dia in model.dias
        )
        # retorno_esperado = model.theta * model.r
        # A dúvida de qual retorno usar (Retorno BRUTO ou Excesso (retorno - selic))

        score_total = (1 - model.theta) * sum(
            model.x[a] * model.score[a] for a in model.ativos
        )
        return retorno_esperado + score_total
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.maximize)
    #=============================
    # RESTRIÇÕES
    #=============================
    def def_r(model):
        return model.r == sum(model.excesso[a]*model.x[a]for a in model.ativos)
    model.const_def_r = pyo.Constraint(rule=def_r)
    ## modelos de Programação de Cone de Segunda Ordem (SOCP) 
    ## e Programação Quadrática com Restrições (QCP)
    def cone(model):
        return model.r**2 >= model.s**2 * sum(model.x[a]*model.sigma[a,b]*model.x[b] for a in model.ativos for b in model.ativos)
    model.constr_cone = pyo.Constraint(rule=cone)
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)
    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)
    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)
    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)
    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)
    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)
    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')
    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    s_lo = 0.01        # <-- o Sharpe DIÁRIO 
    s_hi = 1   # teto: 2x o melhor ativo individual
    tol  = 0.001
    print("MODEL.S.VALUE => ",model.s.value)
    melhor_pesos = None
    historico = []
    carteiras_criadas = []
    # print(f"Valor a ser batido com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
    print(f"Começando o WHILE do ano {ano}")
    while s_hi - s_lo > tol:
        s_a_ser_usado = 0.5 * (s_lo + s_hi)
        model.s.value = s_a_ser_usado
        model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})

        print("="*18)
        res = opt.solve(model, load_solutions=False, tee=False)
        tc = res.solver.termination_condition
        print(f"Modelo Resolveu....Condição: ",tc)
        if tc == pyo.TerminationCondition.optimal:
            model.solutions.load_from(res)
            melhor_pesos = {a: pyo.value(model.x[a]) for a in model.ativos}
            s_lo = model.s.value
            print(f"Valor encontrado para o Sharpe: ",s_lo)
            historico.append((s_a_ser_usado, 'viavel'))
        elif tc in (pyo.TerminationCondition.infeasible,
                    pyo.TerminationCondition.infeasibleOrUnbounded, pyo.TerminationCondition.unbounded,pyo.TerminationCondition.unknown):
            s_hi = s_a_ser_usado
            print(f"Atualização do S_HI : {s_hi}")
            print("="*28)
            historico.append((s_a_ser_usado, 'inviavel'))
        else:
            print(f'status inesperado em s={s_a_ser_usado:.4f}: {tc}')
            pass
    print("SAIU DO WHILE")
    print(f'Sharpe máximo (diário) ≈ {s_lo:.4f}  |  anualizado ≈ {s_lo*np.sqrt(252):.2f}')
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass

=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2025
Nao consta model
Score Atualizado
Retornos atualizados
Excessos Atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2026
## UTILIZANDO SCORE DO ANO DE:  2025
## UTILIZANDO DADOS DE RETORNO DE:  2025
## UTILIZANDO EXCESSO DO ANO DE:  2025
## UTILIZANDO SIGMAS DO ANO DE:  2025
70
70
70
70
MODEL.S.VALUE =>  2
Começando o WHILE do ano 2025
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.505
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.2575
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.38125
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.4431

In [64]:
carteiras_anuais

{2026: {'pesos': {0: 0.15919821525601546,
   1: 1.332914160911375e-10,
   2: 6.049270150771435e-11,
   3: 9.239982613526304e-11,
   4: 4.406373216192304e-11,
   5: 1.3189011208021934e-10,
   6: 2.908023814903517e-10,
   7: 3.1786605775710817e-11,
   8: 0.0,
   9: 1.0227901314444976e-10,
   10: 3.050936056508452e-10,
   11: 1.285762281444118e-10,
   12: 5.6863118723030225e-11,
   13: 4.018713519726888e-10,
   14: 2.3625786471650625e-10,
   15: 3.054041613404318e-11,
   16: 0.19999999620553435,
   17: 3.0797539251597653e-11,
   18: 5.4743176178064496e-11,
   19: 0.02183999633991567,
   20: 4.998742416145063e-11,
   21: 5.5029024073695306e-11,
   22: 4.912153109960645e-11,
   23: 1.0187501680026052e-10,
   24: 9.747203815181747e-11,
   25: 0.0,
   26: 8.48564534931576e-11,
   27: 1.0120489774298435e-10,
   28: 1.0111051812973966e-10,
   29: 7.06965645655839e-11,
   30: 1.5002972147442912e-10,
   31: 1.6758779273047405e-10,
   32: 6.875355665973299e-11,
   33: 9.046382366415212e-11,
   34:

In [68]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': lista_ativos_finais[an-1][k], 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
2016


In [69]:
df_portfolios

,ano,ativo,peso
0,2026,ABEV3,0.1592
1,2026,CSMG3,0.2000
2,2026,CXSE3,0.0218
3,2026,PETR3,0.0200
4,2026,PETR4,0.1353
...,...,...,...
102,2016,MGLU3,0.0320
103,2016,MRVE3,0.1325
104,2016,MULT3,0.2000
105,2016,RADL3,0.2000


In [70]:
df_portfolios.to_csv('carteiras_mf.csv')
